# JRDB RA-L tracker figures — lightweight plotting notebook
This notebook only loads saved tables produced by
`tracker-eval-protocol`. It performs lightweight aggregation and
plotting; it does not run TrackEval, trackers, or scan tracker outputs.

The 2×2 capability figure now includes initialization/re-acquisition as panel (d).
That panel uses a tracker-common event set: a trajectory is established at the
earliest output beginning within 0.5 s that then remains continuously and
correctly assigned with one tracker ID for 1 s. Runtime versus detections is
exported as a separate 1×1 figure.


In [ ]:
# Cell 1 — Importing packages
from pathlib import Path
import json
import os
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/jrdb_ral_matplotlib")

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:  # Allows non-notebook smoke tests.
    def display(value):
        print(value)




In [ ]:
# Cell 2 — Constants and editable plotting configuration
RESULTS_DIR = Path(
    os.environ.get("JRDB_RAL_RESULTS_DIR", "./ral_tracker_results")
).expanduser().resolve()
TABLES_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "notebook_figures"

REFERENCE_TRACKER = "pedreftrack__global_gt_assisted"

# ---------------------------------------------------------------------------
# Plot membership
# ---------------------------------------------------------------------------
# Toggle the GT-assisted reference in non-runtime comparison panels. Runtime
# panels always exclude the reference tracker, irrespective of this flag.
SHOW_REFERENCE_IN_NON_RUNTIME_PLOTS = True
# Accuracy-runtime overview options.
SHOW_REFERENCE_HOTA_LINES_IN_RUNTIME = True
SHOW_ACCURACY_RUNTIME_TRACKER_NAMES = False

# PROFILE_TRACKERS is the editable tracker registry for capability comparisons
# and is deliberately independent of frame_stats availability.
PROFILE_TRACKERS = [
    "pedreftrack__global_gt_assisted",
    "pedreftrack__global_no_gt",
    "fastpoly__global",
    "simpletrack__global",
    "elptnet__global",
    "gnnpmb__global",
    "cbmot__global",
    "ab3dmot__global",
]

COMPARISON_TRACKERS = [
    tracker for tracker in PROFILE_TRACKERS
    if tracker != REFERENCE_TRACKER
]
CAPABILITY_TRACKERS = list(
    dict.fromkeys(
        ([REFERENCE_TRACKER] if SHOW_REFERENCE_IN_NON_RUNTIME_PLOTS else [])
        + COMPARISON_TRACKERS
    )
)

# Runtime plots intentionally exclude the GT-assisted reference.
GLOBAL_RUNTIME_TRACKER_CANDIDATES = COMPARISON_TRACKERS.copy()
EDGE_RUNTIME_TRACKER_CANDIDATES = COMPARISON_TRACKERS.copy()

# Non-runtime comparison plots follow the reference-display flag.
GLOBAL_DELTA_TRACKERS = CAPABILITY_TRACKERS.copy()
INITIALIZATION_TRACKERS = CAPABILITY_TRACKERS.copy()
INITIALIZATION_EVENT_TYPE = "all"  # all | trajectory_start | post_gap_reacquisition

# ---------------------------------------------------------------------------
# Canonical tracker styles
# ---------------------------------------------------------------------------
TRACKER_STYLES = {
    "pedreftrack__global_gt_assisted": {
        "label": "GT-assisted PedRefTrack",
        "color": "#054D6B",
        "linestyle": "-",
    },
    "pedreftrack__global_no_gt": {
        "label": "PedRefTrack (no GT)",
        "color": "#38A4DC",
        "linestyle": "--",
    },
    "fastpoly__global": {
        "label": "Fast-Poly",
        "color": "#F4A641",
        "linestyle": "--",
    },
    "simpletrack__global": {
        "label": "SimpleTrack",
        "color": "#009E73",
        "linestyle": "--",
    },
    "elptnet__global": {
        "label": "ELPTNet (box)",
        "color": "#D55E00",
        "linestyle": "--",
    },
    "gnnpmb__global": {
        "label": "GNN-PMB",
        "color": "#CC79A7",
        "linestyle": "--",
    },
    "cbmot__global": {
        "label": "CBMOT",
        "color": "#7A5195",
        "linestyle": "--",
    },
    "ab3dmot__global": {
        "label": "AB3DMOT",
        "color": "#879AA3",
        "linestyle": "--",
    },
}

TRACKER_LABELS = {
    tracker: style["label"] for tracker, style in TRACKER_STYLES.items()
}
TRACKER_COLORS = {
    tracker: style["color"] for tracker, style in TRACKER_STYLES.items()
}
TRACKER_LINESTYLES = {
    tracker: style["linestyle"] for tracker, style in TRACKER_STYLES.items()
}
COMMON_MARKER = "o"

FPS = 15.0
REALTIME_FPS = 10.0
DESIRED_MAX_GAP_S = 2.0
MIN_GAP_EVENTS_PER_POINT = 100
MIN_NN_CHUNKS_PER_BIN = 100
RUNTIME_MAX_DETECTIONS = 80
RUNTIME_BIN_WIDTH = 5
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = False

PAGE_WIDTH_IN = 7.0

# Requested fixed label placement in the accuracy–runtime panel.
ANNOTATION_PLACEMENT = {
    "simpletrack__global": (-5, 5, "right", "bottom"),
    "gnnpmb__global": (-5, -5, "right", "top"),
    "elptnet__global": (-5, -5, "right", "top"),
    "cbmot__global": (-5, -5, "right", "top"),
    "ab3dmot__global": (5, -5, "left", "top"),
    "pedreftrack__global_no_gt": (5, 5, "left", "bottom"),
    "pedreftrack__global_gt_assisted": (5, -5, "left", "top"),
}

# SCAI white-theme chrome, adapted from tracker evaluation style.
SCAI = {
    "bg": "#FFFFFF",
    "ax_bg": "#FAFBFC",
    "spine": "#D0D0D0",
    "spine_dark": "#333333",
    "text": "#111111",
    "text_dim": "#666666",
    "control": "#1A1A1A",
    "brand_blue": "#1478C8",
}


In [ ]:
# Cell 3 — Non-plotting helper functions
def tracker_label(tracker):
    return TRACKER_LABELS.get(
        tracker, tracker.replace("__global", "").replace("_", " ")
    )


def load_table(filename, required_columns=()):
    path = TABLES_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Check RESULTS_DIR or rerun the extraction script."
        )
    frame = pd.read_csv(path)
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise KeyError(f"{filename} lacks required columns: {sorted(missing)}")
    return frame


def coerce_boolean_series(series):
    """Read bool columns robustly from saved CSV tables."""
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )


def validate_tracker_configuration(*tracker_groups):
    """Fail early when an enabled tracker lacks a prepared style slot."""
    enabled = {
        tracker
        for group in tracker_groups
        for tracker in group
    }
    missing = sorted(enabled - set(TRACKER_STYLES))
    if missing:
        raise KeyError(
            "Enabled trackers without TRACKER_STYLES entries: "
            + ", ".join(missing)
        )


def runtime_bin_bounds(count):
    """Use 0 as its own bin, then 1–5, 6–10, ..., 76–80."""
    count = int(count)
    if count == 0:
        return 0, 0, 0.0
    upper = int(np.ceil(count / RUNTIME_BIN_WIDTH) * RUNTIME_BIN_WIDTH)
    lower = upper - RUNTIME_BIN_WIDTH + 1
    return lower, upper, 0.5 * (lower + upper)


def aggregate_runtime_frames(frame):
    frame = frame[
        (frame["num_det_in"] >= 0)
        & (frame["num_det_in"] <= RUNTIME_MAX_DETECTIONS)
    ].copy()
    if frame.empty:
        return pd.DataFrame()
    bounds = frame["num_det_in"].map(runtime_bin_bounds)
    frame[["bin_left", "bin_right", "bin_mid"]] = pd.DataFrame(
        bounds.tolist(), index=frame.index
    )
    rows = []
    for (tracker, left, right, middle), group in frame.groupby(
        ["tracker", "bin_left", "bin_right", "bin_mid"], sort=True
    ):
        rows.append(
            {
                "tracker": tracker,
                "label": tracker_label(tracker),
                "bin_left": int(left),
                "bin_right": int(right),
                "bin_mid": float(middle),
                "fps_median": float(group["fps"].median()),
                "fps_q10": float(group["fps"].quantile(0.10)),
                "fps_q90": float(group["fps"].quantile(0.90)),
                "n_frames": int(len(group)),
            }
        )
    return pd.DataFrame(rows)


def contiguous_supported_rows(frame, max_x, minimum_count):
    """Retain the initial contiguous bins/ages with adequate common support."""
    ordered = frame.sort_values("bin_index").copy()
    kept = []
    for row in ordered.itertuples(index=False):
        if float(row.x) > max_x + 1e-12:
            break
        if int(row.n_eligible) < minimum_count:
            break
        kept.append(int(row.bin_index))
    return ordered[ordered["bin_index"].isin(kept)].copy()


def contiguous_high_nn_bins(frame, minimum_count):
    """Select a contiguous range downwards from 1.0 m using common support."""
    ordered = frame.sort_values("bin_index")
    selected = []
    for row in ordered.iloc[::-1].itertuples(index=False):
        if int(row.n_eligible) < minimum_count:
            break
        selected.append(int(row.bin_index))
    return sorted(selected)


def assert_common_denominators(frame):
    pivot = frame.pivot_table(
        index=["profile", "bin_index"],
        columns="tracker",
        values="n_eligible",
        aggfunc="first",
    )
    inconsistent = pivot[pivot.nunique(axis=1, dropna=False) != 1]
    if not inconsistent.empty:
        raise RuntimeError(
            "Tracker-independent denominator check failed for "
            f"{len(inconsistent)} profile bins."
        )
    return pivot


def nonnegative_asymmetric_errors(center, lower, upper):
    center = np.asarray(center, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    return np.vstack(
        [np.maximum(0.0, center - lower), np.maximum(0.0, upper - center)]
    )





def assert_initialization_common_denominators(frame):
    """Verify one shared event denominator for every tracker and frame."""
    pivot = frame.pivot_table(
        index=["event_type", "elapsed_frames"],
        columns="tracker",
        values="n_eligible",
        aggfunc="first",
    )
    inconsistent = pivot[pivot.nunique(axis=1, dropna=False) != 1]
    if not inconsistent.empty:
        raise RuntimeError(
            "Initialization common-denominator check failed for "
            f"{len(inconsistent)} rows."
        )
    return pivot


In [ ]:
# Cell 4 — Plotting functions
def apply_scai_white_style():
    mpl.rcParams.update(
        {
            "figure.facecolor": SCAI["bg"],
            "axes.facecolor": SCAI["ax_bg"],
            "axes.edgecolor": SCAI["spine_dark"],
            "axes.labelcolor": SCAI["text"],
            "axes.linewidth": 0.8,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": False,
            "xtick.color": SCAI["text"],
            "ytick.color": SCAI["text"],
            "xtick.major.width": 0.8,
            "ytick.major.width": 0.8,
            "font.family": "sans-serif",
            "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
            "font.size": 9.0,
            "axes.labelsize": 9.0,
            "xtick.labelsize": 8.0,
            "ytick.labelsize": 8.0,
            "legend.fontsize": 8.0,
            "legend.frameon": True,
            "legend.facecolor": SCAI["ax_bg"],
            "legend.edgecolor": SCAI["spine"],
            "lines.linewidth": 2.0,
            "lines.markersize": 5.0,
            "lines.solid_capstyle": "round",
            "patch.linewidth": 0.8,
            "savefig.dpi": 300,
            # "savefig.bbox": "tight",
            "savefig.facecolor": SCAI["bg"],
            "pdf.fonttype": 42,
            "svg.fonttype": "none",
        }
    )


def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(SCAI["spine_dark"])
    ax.spines["bottom"].set_color(SCAI["spine_dark"])


def add_below_caption(ax, letter, title, y=-0.34):
    ax.text(
        0.5,
        y,
        f"({letter}) {title}",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=9,
        # fontweight="bold",
        color=SCAI["text"],
        clip_on=False,
    )


def save_and_display(fig, basename):
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    if SAVE_PNG:
        fig.savefig(FIGURE_DIR / f"{basename}.png", dpi=300)
    if SAVE_PDF:
        fig.savefig(FIGURE_DIR / f"{basename}.pdf")
    if SAVE_SVG:
        fig.savefig(FIGURE_DIR / f"{basename}.svg")
    display(fig)
    plt.close(fig)


def plot_global_hota_runtime(global_table, delta_table):
    apply_scai_white_style()
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(PAGE_WIDTH_IN, PAGE_WIDTH_IN * 0.4),
        dpi=300,
        gridspec_kw={"width_ratios": [1.10, 0.90]},
    )

    # ------------------------------------------------------------------
    # Global HOTA vs tracker-step runtime.
    # The GT-assisted reference is intentionally omitted from this panel.
    # ------------------------------------------------------------------
    ax = axes[0]
    ax.axvline(
        REALTIME_FPS,
        color=SCAI["brand_blue"],
        linestyle="--",
        linewidth=1.4,
        label=f"Real-time ({REALTIME_FPS:g} Hz)",
    )

    # GT-assisted PedRefTrack reference:
    # show its global HOTA and sequence P10-P90 as horizontal lines,
    # but do not plot it as a runtime point.
    if SHOW_REFERENCE_HOTA_LINES_IN_RUNTIME:
        reference_rows = global_table[
            global_table["tracker"] == REFERENCE_TRACKER
        ]

        if reference_rows.empty:
            warnings.warn(
                f"No global HOTA row available for reference tracker "
                f"{REFERENCE_TRACKER}"
            )
        else:
            reference = reference_rows.iloc[0]
            reference_color = TRACKER_COLORS[REFERENCE_TRACKER]

            # Global HOTA.
            ax.axhline(
                float(reference["hota_combined"]),
                color=reference_color,
                linestyle="--",
                linewidth=1.4,
                label=tracker_label(REFERENCE_TRACKER),
                zorder=1,
            )

            # Sequence P10.
            ax.axhline(
                float(reference["hota_seq_qlo"]),
                color=reference_color,
                linestyle=":",
                linewidth=1.0,
                alpha=0.75,
                zorder=1,
            )

            # Sequence P90.
            ax.axhline(
                float(reference["hota_seq_qhi"]),
                color=reference_color,
                linestyle=":",
                linewidth=1.0,
                alpha=0.75,
                label="PedRefTrack sequence P10–P90",
                zorder=1,
            )

    for tracker in GLOBAL_RUNTIME_TRACKERS_AVAILABLE:
        row = global_table[global_table["tracker"] == tracker]
        if row.empty:
            warnings.warn(f"No global HOTA/runtime row for {tracker}")
            continue
        row = row.iloc[0]
        if not np.isfinite(float(row.get("fps_median", np.nan))):
            continue
        x = float(row["fps_median"])
        y = float(row["hota_combined"])
        xerr = nonnegative_asymmetric_errors(
            [x], [float(row["fps_qlo"])], [float(row["fps_qhi"])]
        )
        yerr = nonnegative_asymmetric_errors(
            [y],
            [float(row["hota_seq_qlo"])],
            [float(row["hota_seq_qhi"])],
        )
        ax.errorbar(
            x,
            y,
            xerr=xerr,
            yerr=yerr,
            fmt=COMMON_MARKER,
            markersize=6,
            color=TRACKER_COLORS[tracker],
            ecolor=TRACKER_COLORS[tracker],
            elinewidth=1.2,
            capsize=3.5,
            capthick=1.2,
            zorder=3,
        )
        if SHOW_ACCURACY_RUNTIME_TRACKER_NAMES:
            dx, dy, ha, va = ANNOTATION_PLACEMENT.get(
                tracker, (5, 5, "left", "bottom")
            )
            ax.annotate(
                tracker_label(tracker),
                (x, y),
                xytext=(dx, dy),
                textcoords="offset points",
                ha=ha,
                va=va,
                fontsize=7.5,
                color=SCAI["text"],
            )

    ax.set_xscale("log")
    ax.set_xlabel("Tracker-step FPS")
    ax.set_ylabel("Global HOTA (%)")
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(1.0, 1.03),
        ncol=3,
        handlelength=2.2,
        columnspacing=1.0,
    )
    despine(ax)
    add_below_caption(ax, "a", "Accuracy–runtime overview", y=-0.31)

    # ------------------------------------------------------------------
    # Remaining tracker-side gap. The reference, when enabled, is shown
    # only as its colored zero-gap marker.
    # ------------------------------------------------------------------
    ax = axes[1]
    subset = delta_table[
        delta_table["tracker"].isin(GLOBAL_DELTA_TRACKERS)
    ].copy()
    subset = subset.sort_values("hota_gap_combined").reset_index(drop=True)
    positions = np.arange(len(subset))
    for position, row in zip(positions, subset.itertuples(index=False)):
        center = float(row.hota_gap_combined)
        if row.tracker == REFERENCE_TRACKER:
            ax.plot(
                center,
                position,
                marker=COMMON_MARKER,
                linestyle="None",
                markersize=6,
                color=TRACKER_COLORS[row.tracker],
                zorder=3,
            )
            continue
        error = nonnegative_asymmetric_errors(
            [center],
            [float(row.paired_seq_gap_qlo)],
            [float(row.paired_seq_gap_qhi)],
        )
        ax.errorbar(
            center,
            position,
            xerr=error,
            fmt=COMMON_MARKER,
            markersize=6,
            color=TRACKER_COLORS[row.tracker],
            ecolor=TRACKER_COLORS[row.tracker],
            elinewidth=1.3,
            capsize=3.5,
            capthick=1.2,
            zorder=3,
        )
    ax.axvline(0, color=SCAI["control"], linestyle="--", linewidth=1.0)
    ax.set_yticks(positions, [tracker_label(t) for t in subset["tracker"]])
    ax.set_xlabel("ΔHOTA to GT-assisted PedRefTrack (pp)")
    ax.xaxis.set_label_coords(0.44, -0.16)
    despine(ax)
    add_below_caption(ax, "b", "Remaining tracker-side gap", y=-0.31)

    fig.subplots_adjust(
        top=0.84,
        bottom=0.25,
        left=0.08,
        right=0.98,
        wspace=0.55,
    )
    return fig

def plot_profile_curve(
    ax,
    frame,
    tracker,
    y_column,
    lower_column,
    upper_column,
):
    subset = frame[frame["tracker"] == tracker].sort_values("x")
    if subset.empty:
        return
    color = TRACKER_COLORS[tracker]
    linestyle = TRACKER_LINESTYLES.get(tracker, "-")
    ax.fill_between(
        subset["x"].to_numpy(float),
        subset[lower_column].to_numpy(float),
        subset[upper_column].to_numpy(float),
        color=color,
        alpha=0.16,
        linewidth=0,
    )
    ax.plot(
        subset["x"],
        subset[y_column],
        color=color,
        linestyle=linestyle,
        marker=COMMON_MARKER,
        label=tracker_label(tracker),
    )


def plot_capability_stratification(
    gap_frame,
    recovery_frame,
    nn_failure_frame,
    initialization_frame,
    initialization_event_type,
    gap_max_s,
    recovery_max_s,
    nn_edges,
):
    apply_scai_white_style()
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(PAGE_WIDTH_IN, PAGE_WIDTH_IN*0.60),
        dpi=300,
    )

    ax = axes[0, 0]
    for tracker in CAPABILITY_TRACKERS:
        plot_profile_curve(
            ax,
            gap_frame,
            tracker,
            "success_percent",
            "ci_lo_percent",
            "ci_hi_percent",
        )
    ax.set_xlim(0, gap_max_s + 0.02)
    ax.set_xticks(np.arange(0, gap_max_s + 0.001, 0.5))
    ax.set_ylim(0, 101)
    ax.set_xlabel("Detector-gap age (s)")
    ax.set_ylabel("Tracking success (%)")
    despine(ax)
    add_below_caption(ax, "a", "Tracking in the gap", y=-0.39)

    ax = axes[0, 1]
    for tracker in CAPABILITY_TRACKERS:
        plot_profile_curve(
            ax,
            recovery_frame,
            tracker,
            "success_percent",
            "ci_lo_percent",
            "ci_hi_percent",
        )
    ax.set_xlim(0, recovery_max_s + 0.02)
    ax.set_xticks(np.arange(0, recovery_max_s + 0.001, 0.5))
    ax.set_ylim(0, 101)
    ax.set_xlabel("Detector-gap duration (s)")
    ax.set_ylabel("Post-gap recovery\nsuccess (%)")
    despine(ax)
    add_below_caption(ax, "b", "Post-gap identity recovery", y=-0.39)

    ax = axes[1, 0]
    for tracker in CAPABILITY_TRACKERS:
        plot_profile_curve(
            ax,
            nn_failure_frame,
            tracker,
            "failure_percent",
            "failure_ci_lo_percent",
            "failure_ci_hi_percent",
        )
    ax.set_xticks(nn_edges)
    ax.set_xlim(float(nn_edges[0]), float(nn_edges[-1]))
    ax.set_ylim(0, 30)
    ax.set_xlabel("Nearest-neighbour distance (m)")
    ax.set_ylabel("Association failure (%)")
    despine(ax)
    add_below_caption(
        ax, "c", "Nearest-neighbour association stratification", y=-0.39
    )

    # Track initialization / re-acquisition replaces the former runtime panel.
    ax = axes[1, 1]
    for tracker in INITIALIZATION_TRACKERS_AVAILABLE:
        subset = initialization_frame[
            (initialization_frame["tracker"] == tracker)
            & (initialization_frame["event_type"] == initialization_event_type)
        ].sort_values("elapsed_seconds")
        if subset.empty:
            continue
        x = subset["elapsed_seconds"].to_numpy(float)
        center = subset["stable_established_percent"].to_numpy(float)
        lower = subset["ci_lo_percent"].to_numpy(float)
        upper = subset["ci_hi_percent"].to_numpy(float)
        color = TRACKER_COLORS[tracker]
        ax.fill_between(
            x,
            lower,
            upper,
            color=color,
            alpha=0.12,
            linewidth=0,
        )
        ax.plot(
            x,
            center,
            color=color,
            linestyle=TRACKER_LINESTYLES.get(tracker, "-"),
            marker=COMMON_MARKER,
            label=tracker_label(tracker),
        )

    event_rows = initialization_frame[
        initialization_frame["event_type"] == initialization_event_type
    ]
    max_time = float(event_rows["elapsed_seconds"].max())
    min_time = float(event_rows["elapsed_seconds"].min())
    padding = max(0.01, 0.035 * max_time)
    ax.set_xlim(max(0.0, min_time - padding), max_time + padding)
    ax.set_xticks(np.arange(0.1, 0.51, 0.1))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
    ax.set_ylim(0, 70)
    ax.set_xlabel("Time since detector support (s)")
    ax.set_ylabel("Stable trajectories\nestablished (%)")
    despine(ax)
    add_below_caption(
        ax,
        "d",
        "Track initialization",
        y=-0.39,
    )

    handles = [
        Line2D(
            [0],
            [0],
            color=TRACKER_COLORS[tracker],
            linestyle=TRACKER_LINESTYLES.get(tracker, "-"),
            marker=COMMON_MARKER,
            label=tracker_label(tracker),
        )
        for tracker in CAPABILITY_TRACKERS
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.995),
        ncol=max(1, int(np.ceil(len(CAPABILITY_TRACKERS) / 2))),
        columnspacing=0.9,
        handlelength=1.8,
        fontsize=8.0,
    )
    fig.subplots_adjust(
        top=0.88,
        bottom=0.16,
        left=0.09,
        right=0.98,
        hspace=0.60,
        wspace=0.40,
    )
    return fig

def plot_coverage_diagnostics(
    gap_support,
    nn_support,
    runtime_binned,
    gap_max_s,
    nn_edges,
):
    apply_scai_white_style()

    fig, axes = plt.subplots(
        3,
        1,
        dpi=300,
        figsize=(PAGE_WIDTH_IN / 2, PAGE_WIDTH_IN / 1.90),
    )

    def apply_log_y_axis(ax):
        """Use a logarithmic y-axis with ticks formatted as 10^x."""
        ax.set_yscale("log", nonpositive="clip")
        ax.yaxis.set_major_locator(mticker.LogLocator(base=10))
        ax.yaxis.set_major_formatter(mticker.LogFormatterMathtext(base=10))

    # ------------------------------------------------------------------
    # Detector-gap coverage
    # ------------------------------------------------------------------
    ax = axes[0]

    # post_gap_recovery x_left/x_right are stored in frames.
    # Convert the actual bin boundaries to seconds.
    gap_left_s = gap_support["x_left"].to_numpy(float) / FPS
    gap_right_s = gap_support["x_right"].to_numpy(float) / FPS
    gap_width_s = gap_right_s - gap_left_s

    ax.bar(
        gap_left_s,
        gap_support["n_eligible"].to_numpy(float),
        width=gap_width_s,
        align="edge",
        color="#C8DFFA",
        edgecolor=SCAI["brand_blue"],
        linewidth=1.1,
    )

    ax.set_xlim(
        0,
        min(DESIRED_MAX_GAP_S, float(gap_right_s.max())) + 0.02,
    )
    ax.set_xlabel("Detector-gap duration (s)")
    ax.set_ylabel("Detection gap \n count")

    apply_log_y_axis(ax)
    
    despine(ax)

    # ------------------------------------------------------------------
    # Nearest-neighbour coverage
    # ------------------------------------------------------------------
    ax = axes[1]

    ax.bar(
        nn_support["x"],
        nn_support["n_eligible"],
        width=0.085,
        color="#C8DFFA",
        edgecolor=SCAI["brand_blue"],
        linewidth=1.1,
    )

    ax.set_xticks(nn_edges)
    ax.set_xlim(float(nn_edges[0]), float(nn_edges[-1]))
    ax.set_xlabel("Nearest-neighbour distance (m)")
    ax.set_ylabel("Nearest-neighbour\ndistance count")

    apply_log_y_axis(ax)

    despine(ax)

    # ------------------------------------------------------------------
    # Runtime-bin coverage
    # ------------------------------------------------------------------
    ax = axes[2]

    if runtime_binned.empty:
        ax.text(
            0.5,
            0.5,
            "No runtime frame statistics available",
            transform=ax.transAxes,
            ha="center",
            va="center",
        )
    else:
        common_runtime = (
            runtime_binned.groupby(
                ["bin_left", "bin_right", "bin_mid"],
                as_index=False,
            )["n_frames"]
            .min()
            .sort_values("bin_mid")
        )

        ax.bar(
            common_runtime["bin_mid"],
            common_runtime["n_frames"],
            width=4.2,
            color="#C8DFFA",
            edgecolor=SCAI["brand_blue"],
            linewidth=1.1,
        )

    ax.set_xlim(0, RUNTIME_MAX_DETECTIONS)
    ax.set_xticks(np.arange(0, RUNTIME_MAX_DETECTIONS + 1, 10))
    ax.set_xlabel("Number of pedestrian detections")
    ax.set_ylabel("Frame count")

    apply_log_y_axis(ax)

    # Avoid an invalid automatic logarithmic range if no runtime data exist.
    if runtime_binned.empty:
        ax.set_ylim(1, 10)

    despine(ax)

    # One shared caption for the complete three-panel figure.
    fig.text(
        0.5,
        0.02,
        "(b) Data coverage histograms",
        ha="center",
        va="bottom",
        # fontweight="bold",
    )

    fig.subplots_adjust(
        top=0.96,
        bottom=0.18,
        left=0.20,
        right=0.97,
        hspace=0.6,
    )

    return fig





def plot_runtime_vs_detections(runtime_frame):
    """Plot tracker-step FPS versus the number of input detections."""
    apply_scai_white_style()    
    fig, ax = plt.subplots(
        1,
        1,
        dpi=300,
        figsize=(PAGE_WIDTH_IN / 2, PAGE_WIDTH_IN / 2.5),
    )

    for tracker in EDGE_RUNTIME_TRACKERS_AVAILABLE:
        subset = runtime_frame[
            runtime_frame["tracker"] == tracker
        ].sort_values("bin_mid")
        if subset.empty:
            continue
        error = nonnegative_asymmetric_errors(
            subset["fps_median"],
            subset["fps_q10"],
            subset["fps_q90"],
        )
        ax.errorbar(
            subset["bin_mid"],
            subset["fps_median"],
            yerr=error,
            color=TRACKER_COLORS[tracker],
            ecolor=TRACKER_COLORS[tracker],
            linestyle=TRACKER_LINESTYLES.get(tracker, "-"),
            marker=COMMON_MARKER,
            elinewidth=1.0,
            capsize=2.5,
            capthick=1.0,
            label=tracker_label(tracker),
        )

    ax.axhline(
        REALTIME_FPS,
        color=SCAI["control"],
        linestyle="--",
        linewidth=1.0,
    )
    ax.set_xlim(0, RUNTIME_MAX_DETECTIONS)
    ax.set_xticks(np.arange(0, RUNTIME_MAX_DETECTIONS + 1, 10))
    ax.set_yscale("log")
    ax.set_xlabel("Number of input detections")
    ax.set_ylabel("Tracker-step FPS")
    despine(ax)
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.42, 1.02),
        ncol=3,
        columnspacing=0.7,
        handlelength=1.5,
        fontsize=8.0,
    )
    # add_below_caption(
    #     ax,
    #     "a",
    #     "Runtime vs. detections per frame",
    #     y=-0.35,
    # )
    fig.subplots_adjust(
        top=0.90,
        bottom=0.12,
        left=0.10,
        right=0.98,
    )
    return fig



In [ ]:
# Cell 5 — Loading and preparing all data
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

validate_tracker_configuration(
    PROFILE_TRACKERS,
    CAPABILITY_TRACKERS,
    GLOBAL_RUNTIME_TRACKER_CANDIDATES,
    GLOBAL_DELTA_TRACKERS,
    EDGE_RUNTIME_TRACKER_CANDIDATES,
    INITIALIZATION_TRACKERS,
    [REFERENCE_TRACKER],
)

# HOTA is loaded independently from timing, then exact per-variant timing is
# merged below. This prevents a mapped/shared timing source from leaking into
# the global HOTA–FPS panel.
global_hota_metrics = load_table(
    "global_hota_metrics.csv",
    required_columns=(
        "tracker",
        "hota_combined",
        "hota_seq_qlo",
        "hota_seq_qhi",
    ),
)
hota_delta = load_table(
    "hota_gap_to_reference.csv",
    required_columns=(
        "tracker",
        "hota_gap_combined",
        "paired_seq_gap_qlo",
        "paired_seq_gap_qhi",
    ),
)
direct_profiles = load_table(
    "direct_profile_summary.csv",
    required_columns=(
        "tracker",
        "profile",
        "bin_index",
        "x",
        "x_left",
        "x_right",
        "n_eligible",
        "n_success",
        "success_percent",
        "ci_lo_percent",
        "ci_hi_percent",
    ),
)
denominator_audit = assert_common_denominators(direct_profiles)

# ---------------------------------------------------------------------------
# Exact per-variant frame_stats from saved computation outputs
# ---------------------------------------------------------------------------
frame_stats_availability = load_table(
    "frame_stats_availability.csv",
    required_columns=(
        "tracker",
        "runtime_source_tracker",
        "uses_mapped_source",
        "has_usable_frame_stats",
        "status",
        "n_runtime_frames",
    ),
)
saved_runtime_summary = load_table(
    "runtime_summary.csv",
    required_columns=(
        "tracker",
        "runtime_source_tracker",
        "fps_median",
        "fps_qlo",
        "fps_qhi",
        "n_runtime_frames",
    ),
)
saved_runtime_frames = load_table(
    "runtime_frame_stats.csv.gz",
    required_columns=(
        "tracker",
        "runtime_source_tracker",
        "step_ms",
        "fps",
        "num_det_in",
    ),
)

has_usable = coerce_boolean_series(
    frame_stats_availability["has_usable_frame_stats"]
)
uses_mapped_source = coerce_boolean_series(
    frame_stats_availability["uses_mapped_source"]
)
same_availability_source = (
    frame_stats_availability["tracker"].astype(str)
    == frame_stats_availability["runtime_source_tracker"].astype(str)
)

# A tracker is timing-eligible only when the saved computation found usable
# frame_stats under that exact variant name. Explicit source mappings are
# rejected even if they exist in the saved result tables.
exact_availability = frame_stats_availability[
    has_usable
    & ~uses_mapped_source
    & same_availability_source
].copy()
exact_availability_trackers = set(
    exact_availability["tracker"].astype(str)
)

exact_runtime_summary = saved_runtime_summary[
    saved_runtime_summary["tracker"].astype(str).isin(
        exact_availability_trackers
    )
    & (
        saved_runtime_summary["tracker"].astype(str)
        == saved_runtime_summary["runtime_source_tracker"].astype(str)
    )
].copy()
exact_runtime_summary = exact_runtime_summary.drop_duplicates(
    subset=["tracker"],
    keep="last",
)

exact_runtime_frames = saved_runtime_frames[
    saved_runtime_frames["tracker"].astype(str).isin(
        exact_availability_trackers
    )
    & (
        saved_runtime_frames["tracker"].astype(str)
        == saved_runtime_frames["runtime_source_tracker"].astype(str)
    )
].copy()
exact_runtime_frames["fps"] = pd.to_numeric(
    exact_runtime_frames["fps"],
    errors="coerce",
)
exact_runtime_frames["num_det_in"] = pd.to_numeric(
    exact_runtime_frames["num_det_in"],
    errors="coerce",
)

global_hota_runtime = global_hota_metrics.merge(
    exact_runtime_summary[
        [
            "tracker",
            "runtime_source_tracker",
            "fps_median",
            "fps_qlo",
            "fps_qhi",
            "n_runtime_frames",
        ]
    ],
    on="tracker",
    how="left",
    validate="one_to_one",
)

exact_fps_trackers = set(
    exact_runtime_summary.loc[
        np.isfinite(
            pd.to_numeric(
                exact_runtime_summary["fps_median"],
                errors="coerce",
            )
        ),
        "tracker",
    ].astype(str)
)
exact_detection_runtime_trackers = set(
    exact_runtime_frames.loc[
        np.isfinite(exact_runtime_frames["fps"])
        & np.isfinite(exact_runtime_frames["num_det_in"]),
        "tracker",
    ].astype(str)
)

GLOBAL_RUNTIME_TRACKERS_AVAILABLE = [
    tracker
    for tracker in GLOBAL_RUNTIME_TRACKER_CANDIDATES
    if tracker in exact_fps_trackers
]
EDGE_RUNTIME_TRACKERS_AVAILABLE = [
    tracker
    for tracker in EDGE_RUNTIME_TRACKER_CANDIDATES
    if tracker in exact_detection_runtime_trackers
]

# Runtime-vs-detections is a separate frame_stats-dependent figure. The 2×2
# capability panels are independent of frame_stats availability.
runtime_frames = exact_runtime_frames[
    exact_runtime_frames["tracker"].isin(
        EDGE_RUNTIME_TRACKERS_AVAILABLE
    )
].copy()
runtime_binned = aggregate_runtime_frames(runtime_frames)

# ---------------------------------------------------------------------------
# Capability-profile populations and supported plotting ranges
# ---------------------------------------------------------------------------
reference_gap_support = direct_profiles[
    (direct_profiles["tracker"] == REFERENCE_TRACKER)
    & (direct_profiles["profile"] == "gap_prediction")
].sort_values("bin_index")
reference_recovery_support = direct_profiles[
    (direct_profiles["tracker"] == REFERENCE_TRACKER)
    & (direct_profiles["profile"] == "post_gap_recovery")
].sort_values("bin_index")
reference_nn_support = direct_profiles[
    (direct_profiles["tracker"] == REFERENCE_TRACKER)
    & (direct_profiles["profile"] == "strict_continuity")
].sort_values("bin_index")

selected_gap_support = contiguous_supported_rows(
    reference_gap_support,
    DESIRED_MAX_GAP_S,
    MIN_GAP_EVENTS_PER_POINT,
)

# Include every calculated recovery-duration bin through 2.0 s.
# Unlike gap-age support, duration-bin support is not cumulative and may
# therefore fall below MIN_GAP_EVENTS_PER_POINT at longer durations.
selected_recovery_support = reference_recovery_support[
    reference_recovery_support["x"]
    <= DESIRED_MAX_GAP_S + 1e-12
].sort_values("bin_index").copy()

if selected_gap_support.empty:
    raise RuntimeError("No detector-gap age reaches the minimum support.")
if selected_recovery_support.empty:
    raise RuntimeError("No post-gap recovery bins are available.")

GAP_PLOT_MAX_S = DESIRED_MAX_GAP_S
RECOVERY_PLOT_MAX_S = DESIRED_MAX_GAP_S

gap_bins = selected_gap_support["bin_index"].astype(int).tolist()
recovery_bins = (
    selected_recovery_support["bin_index"].astype(int).tolist()
)

gap_profiles = direct_profiles[
    (direct_profiles["profile"] == "gap_prediction")
    & (direct_profiles["tracker"].isin(CAPABILITY_TRACKERS))
    & (direct_profiles["bin_index"].isin(gap_bins))
].copy()
recovery_profiles = direct_profiles[
    (direct_profiles["profile"] == "post_gap_recovery")
    & (direct_profiles["tracker"].isin(CAPABILITY_TRACKERS))
    & (direct_profiles["bin_index"].isin(recovery_bins))
].copy()

selected_nn_bins = contiguous_high_nn_bins(
    reference_nn_support, MIN_NN_CHUNKS_PER_BIN
)
if not selected_nn_bins:
    raise RuntimeError(
        "No contiguous nearest-neighbour range has enough chunks."
    )
nn_profiles = direct_profiles[
    (direct_profiles["profile"] == "strict_continuity")
    & (direct_profiles["tracker"].isin(CAPABILITY_TRACKERS))
    & (direct_profiles["bin_index"].isin(selected_nn_bins))
].copy()
nn_profiles["failure_percent"] = 100.0 - nn_profiles["success_percent"]
nn_profiles["failure_ci_lo_percent"] = (
    100.0 - nn_profiles["ci_hi_percent"]
)
nn_profiles["failure_ci_hi_percent"] = (
    100.0 - nn_profiles["ci_lo_percent"]
)

nn_support_selected = reference_nn_support[
    reference_nn_support["bin_index"].isin(selected_nn_bins)
].copy()
NN_EDGES = np.r_[
    nn_support_selected["x_left"].to_numpy(float),
    float(nn_support_selected["x_right"].iloc[-1]),
]


# ---------------------------------------------------------------------------
# Common trajectory-initialization/re-acquisition latency
# ---------------------------------------------------------------------------
initialization_latency = load_table(
    "initialization_latency_summary.csv",
    required_columns=(
        "tracker",
        "event_type",
        "elapsed_frames",
        "elapsed_seconds",
        "n_eligible",
        "n_stably_established_by_time",
        "stable_established_percent",
        "ci_lo_percent",
        "ci_hi_percent",
        "validation_frames",
        "validation_s",
        "required_gt_frames",
    ),
)
initialization_common_events = load_table(
    "initialization_common_events.csv.gz",
    required_columns=(
        "event_id",
        "seq",
        "event_type",
        "anchor_frame",
        "max_start_frames",
        "max_start_s",
        "validation_frames",
        "validation_s",
        "required_gt_frames",
        "n_detector_supported_frames",
        "detector_support_fraction",
    ),
)
initialization_tracker_events = load_table(
    "initialization_tracker_events.csv.gz",
    required_columns=(
        "event_id",
        "seq",
        "event_type",
        "tracker",
        "latency_frames",
        "latency_seconds",
        "stably_established_within_horizon",
        "failure_reason",
    ),
)
initialization_event_audit_summary = load_table(
    "initialization_event_audit_summary.csv",
    required_columns=(
        "event_type",
        "eligible",
        "exclusion_reason",
        "n_candidates",
    ),
)

initialization_denominator_audit = (
    assert_initialization_common_denominators(initialization_latency)
)
initialization_result_trackers = set(
    initialization_latency["tracker"].astype(str)
)
INITIALIZATION_TRACKERS_AVAILABLE = [
    tracker
    for tracker in INITIALIZATION_TRACKERS
    if tracker in initialization_result_trackers
]
if not INITIALIZATION_TRACKERS_AVAILABLE:
    raise RuntimeError(
        "None of INITIALIZATION_TRACKERS is present in "
        "initialization_latency_summary.csv."
    )
if INITIALIZATION_EVENT_TYPE not in set(
    initialization_latency["event_type"].astype(str)
):
    raise ValueError(
        f"Unknown INITIALIZATION_EVENT_TYPE={INITIALIZATION_EVENT_TYPE!r}."
    )


In [ ]:
# Cell 6 — Global HOTA/runtime and ΔHOTA figure
global_figure = plot_global_hota_runtime(
    global_hota_runtime,
    hota_delta,
)
save_and_display(
    global_figure,
    "global_hota_runtime_and_pedreftrack_gap_restyled",
)




In [ ]:
# Cell 7 — Four-panel capability stratification figure
capability_figure = plot_capability_stratification(
    gap_profiles,
    recovery_profiles,
    nn_profiles,
    initialization_latency,
    INITIALIZATION_EVENT_TYPE,
    GAP_PLOT_MAX_S,
    RECOVERY_PLOT_MAX_S,
    NN_EDGES,
)
save_and_display(
    capability_figure,
    "tracker_capability_stratification_2x2_restyled",
)


In [ ]:
# Cell 8 — Coverage diagnostics
coverage_figure = plot_coverage_diagnostics(
    reference_recovery_support,
    reference_nn_support,
    runtime_binned,
    RECOVERY_PLOT_MAX_S,
    np.r_[
        reference_nn_support["x_left"].to_numpy(float),
        float(reference_nn_support["x_right"].iloc[-1]),
    ],
)
save_and_display(
    coverage_figure,
    "stratification_coverage_diagnostics",
)




In [ ]:
# Cell 9 — Runtime scaling with the number of input detections
runtime_figure = plot_runtime_vs_detections(runtime_binned)
save_and_display(
    runtime_figure,
    "runtime_vs_detections_per_frame",
)


In [ ]:
# Cell 10 — Diagnostics and important statistics
print(f"Results directory: {RESULTS_DIR}")
print(f"Figures saved under: {FIGURE_DIR}")
print()
print(
    "Detector-gap horizon available in table: "
    f"{float(reference_gap_support['x'].max()):.2f} s"
)
print(
    f"Detector-gap horizon plotted (N ≥ {MIN_GAP_EVENTS_PER_POINT}): "
    f"{GAP_PLOT_MAX_S:.2f} s"
)
print(
    f"Post-gap recovery midpoint plotted through: "
    f"{RECOVERY_PLOT_MAX_S:.2f} s"
)
if float(reference_gap_support["x"].max()) < DESIRED_MAX_GAP_S - 1e-9:
    print(
        "WARNING: the extracted table stops before 2.0 s. Rerun the Python "
        "extractor with --max-gap-age-s 2.0 and recovery edges through 31."
    )

print()
print("Selected nearest-neighbour bins:")
display(
    nn_support_selected[
        ["x_left", "x_right", "x", "n_eligible"]
    ].reset_index(drop=True)
)

print()
print("Exact per-variant frame_stats availability:")
display(
    frame_stats_availability[
        [
            "tracker",
            "runtime_source_tracker",
            "uses_mapped_source",
            "has_usable_frame_stats",
            "status",
            "n_runtime_frames",
        ]
    ].reset_index(drop=True)
)

print(
    "Omitted from global HOTA–FPS because exact frame_stats are unavailable:"
)
print(
    [
        tracker
        for tracker in GLOBAL_RUNTIME_TRACKER_CANDIDATES
        if tracker not in GLOBAL_RUNTIME_TRACKERS_AVAILABLE
    ]
)
print(
    "Omitted from runtime-vs-detections because exact usable "
    "frame_stats are unavailable:"
)
print(
    [
        tracker
        for tracker in EDGE_RUNTIME_TRACKER_CANDIDATES
        if tracker not in EDGE_RUNTIME_TRACKERS_AVAILABLE
    ]
)

print()
print("Runtime frame counts in the final 76–80 detection bin:")
if runtime_binned.empty:
    print("No exact per-variant runtime rows are available.")
else:
    last_runtime_bin = runtime_binned[
        (runtime_binned["bin_left"] == 76)
        & (runtime_binned["bin_right"] == 80)
    ][["tracker", "label", "n_frames"]]
    display(last_runtime_bin.reset_index(drop=True))

print("Global HOTA and exact runtime inputs:")
display(
    global_hota_runtime[
        global_hota_runtime["tracker"].isin(
            GLOBAL_RUNTIME_TRACKERS_AVAILABLE
        )
    ][
        [
            "tracker",
            "hota_combined",
            "hota_seq_qlo",
            "hota_seq_qhi",
            "fps_median",
            "fps_qlo",
            "fps_qhi",
        ]
    ].reset_index(drop=True)
)

print(
    "Common-denominator audit: "
    f"{len(denominator_audit)} profile bins checked; all consistent."
)
print(
    "All four 2×2 capability panels use CAPABILITY_TRACKERS irrespective "
    "of frame_stats; global HOTA–FPS and the separate runtime-vs-detections "
    "figure require exact per-variant timing and exclude the reference tracker."
)
print(
    "Reminder: panel (c) uses the requested label 'Association failure', "
    "but strict failure also includes missing accepted tracker output."
)


print()
print("Stable initialization/re-acquisition common events:")
display(
    initialization_common_events.groupby(
        "event_type",
        as_index=False,
    ).agg(
        n_events=("event_id", "nunique"),
        n_sequences=("seq", "nunique"),
        max_start_frames=("max_start_frames", "first"),
        validation_frames=("validation_frames", "first"),
        required_gt_frames=("required_gt_frames", "first"),
        median_detector_support_fraction=(
            "detector_support_fraction",
            "median",
        ),
    )
)
print(
    "Initialization plot event type: "
    f"{INITIALIZATION_EVENT_TYPE!r}; plotted trackers: "
    f"{INITIALIZATION_TRACKERS_AVAILABLE}"
)
print(
    "Omitted from initialization plot because results are unavailable:"
)
print(
    [
        tracker
        for tracker in INITIALIZATION_TRACKERS
        if tracker not in INITIALIZATION_TRACKERS_AVAILABLE
    ]
)

print("Final stable-establishment percentage at the latency limit:")
final_initialization = (
    initialization_latency[
        initialization_latency["event_type"] == INITIALIZATION_EVENT_TYPE
    ]
    .sort_values("elapsed_frames")
    .groupby("tracker", as_index=False)
    .tail(1)
)
display(
    final_initialization[
        [
            "tracker",
            "elapsed_frames",
            "elapsed_seconds",
            "n_eligible",
            "n_stably_established_by_time",
            "stable_established_percent",
            "ci_lo_percent",
            "ci_hi_percent",
        ]
    ].reset_index(drop=True)
)

print("Tracker-event failure reasons:")
failure_reason_summary = (
    initialization_tracker_events.groupby(
        ["event_type", "tracker", "failure_reason"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "n_events"})
)
display(failure_reason_summary)

print("Initialization/re-acquisition candidate audit:")
display(initialization_event_audit_summary)
print(
    "Initialization common-denominator audit: "
    f"{len(initialization_denominator_audit)} event-type/time rows checked; "
    "all consistent."
)
